# 06 — Bridge to the agent / RL phase
A decision model is a fast, cheap **router or guard** inside an agent loop: it chooses the next action with a probability, while deterministic code enforces rules and uncertain decisions are escalated to a stronger model (or a human).

Phase 2 of the project frames a small sequential task as an RL-style objective: `reward = task_success - step_cost`. We simulate it with the free mock.


In [ ]:
import random
from decision_qa import MockClient, ChoiceQuestion

client = MockClient(seed=7)
ACTIONS = ['retrieve', 'answer', 'escalate']

def agent_step(state, rng):
    # Simulate a policy that has learned to retrieve before answering.
    target = 0 if not state['has_evidence'] else 1
    q = ChoiceQuestion(label='next_action', options=ACTIONS, gold_index=target)
    result = client.decision(state, [q])[0]
    action = max(result.probabilities, key=result.probabilities.get)
    confidence = result.probabilities[action]
    # deterministic policy: never allow 'answer' without enough evidence
    if action == 'answer' and not state['has_evidence']:
        action = 'retrieve'
    if confidence < 0.6:
        action = 'escalate'
    return action, confidence


In [ ]:
def episode(rng, max_steps=4):
    state = {'question': 'What did the trial report?', 'has_evidence': False}
    steps = 0
    while steps < max_steps:
        action, conf = agent_step(state, rng)
        steps += 1
        if action == 'retrieve':
            state['has_evidence'] = True
        elif action == 'answer':
            return 1.0 - 0.1 * steps  # success minus step cost
        else:
            return -0.2  # escalated: small penalty, no wrong answer
    return -0.1

rng = random.Random(0)
rewards = [episode(rng) for _ in range(20)]
print('mean reward:', round(sum(rewards)/len(rewards), 3))


## Next steps
- Replace the mock with a local open model via `SystemOneClient`.
- Add a real task and reward (e.g. multi-hop QA: reward = exact match - latency cost).
- Study open agent loops such as [JevLoop](https://github.com/zjunlp/JevLoop) and [ProgressGate](https://github.com/AshutoshVJTI/progressgate).

All links and tools used in this repository are free/open.
